# 10 · How good is the "no card" soil fill?

The app fills soil for farmers without a Soil Health Card (`recommend.area_soil`): the district's own SHC average if it
has ≥ 100 tests, else the mean of the 3 nearest such districts. 84 of the app's 623 districts get the nearest fill.

**Test: leave one district out.** For each of the 539 districts that do have an SHC average, pretend they don't: predict
their N, P, K, pH from their nearest *other* districts and compare with their real average.

- **Methods:** k nearest districts (k = 1, 3, 5, 10), 3 nearest in the same state, and two baselines: state mean and
  India mean (both leave the district out).
- **Metrics per nutrient:** mean absolute error (MAE), skill = 1 − MAE / MAE of the India mean (0 = no better than
  one number for all of India, 1 = perfect), correlation, and **class match**: the predicted value falls in the same
  card class as the real one (card limits N 280/560, P 28/56, K 140/280 kg/ha, pH 6.5/8.2; `data.SHC_LEVELS`).
- Caveat: the "real" district values are themselves estimates from SHC class counts (`data.district_soil()`).

In [1]:
import sys
sys.path.insert(0, '../project')
from pathlib import Path

import numpy as np
import pandas as pd

import recommend

RESULTS = Path('../research/results')
NUTRIENTS = ['N', 'P', 'K', 'ph']
LIMITS = {'N': (280, 560), 'P': (28, 56), 'K': (140, 280), 'ph': (6.5, 8.2)}
pd.set_option('display.width', 220)

soil = recommend._soil().reset_index(drop=True)
lat, lon = np.radians(soil.lat.values), np.radians(soil.lon.values)
# great-circle distance in km between every pair of districts
D = 6371 * np.arccos(np.clip(np.sin(lat)[:, None] * np.sin(lat) + np.cos(lat)[:, None] * np.cos(lat) * np.cos(lon[:, None] - lon), -1, 1))
np.fill_diagonal(D, np.inf)  # never use the district itself
same_state = soil.state_name.values[:, None] == soil.state_name.values
print(len(soil), 'districts with an SHC average and coordinates,', soil.state_name.nunique(), 'states')

539 districts with an SHC average and coordinates, 28 states


In [2]:
X = soil[NUTRIENTS].values


def knn(k, mask=None):
    d = np.where(mask, D, np.inf) if mask is not None else D
    idx = np.argsort(d, axis=1)[:, :k]
    pred = X[idx].mean(axis=1)
    alone = ~np.isfinite(np.take_along_axis(d, idx, axis=1)).all(axis=1)  # fewer than k same-state districts
    pred[alone] = np.nan
    return pred


def group_mean(mask):
    w = mask & ~np.eye(len(X), dtype=bool)
    with np.errstate(invalid='ignore'):
        return (w @ X) / w.sum(axis=1, keepdims=True)


preds = {f'{k} nearest': knn(k) for k in (1, 3, 5, 10)}
preds['3 nearest, same state'] = knn(3, same_state)
preds['state mean'] = group_mean(same_state)
preds['India mean'] = group_mean(np.ones_like(same_state))
for p in ('3 nearest, same state', 'state mean'):  # one-district states: fall back to the India mean
    preds[p] = np.where(np.isnan(preds[p]), preds['India mean'], preds[p])


def cls(v, n):
    lo, hi = LIMITS[n]
    return np.digitize(v, [lo, hi])


rows = []
for name, p in preds.items():
    for j, n in enumerate(NUTRIENTS):
        err = np.abs(p[:, j] - X[:, j])
        rows.append({'method': name, 'nutrient': n, 'MAE': err.mean(),
                     'skill': 1 - err.mean() / np.abs(preds['India mean'][:, j] - X[:, j]).mean(),
                     'r': np.corrcoef(p[:, j], X[:, j])[0, 1],
                     'class match %': (cls(p[:, j], n) == cls(X[:, j], n)).mean() * 100})
res = pd.DataFrame(rows)
res.pivot(index='method', columns='nutrient', values=['MAE', 'skill', 'class match %']).reindex(list(preds)).round(2)

MAE                     skill                   class match %                     
nutrient                   K      N      P    ph     K     N     P    ph             K      N      P     ph
method                                                                                                     
1 nearest              33.50  45.35   9.13  0.14  0.20  0.35  0.14  0.49         75.51  72.54  69.57  93.69
3 nearest              28.59  40.75   7.90  0.12  0.32  0.41  0.25  0.56         78.85  73.84  75.88  96.10
5 nearest              27.90  41.68   8.10  0.12  0.33  0.40  0.23  0.55         79.96  75.32  75.88  95.92
10 nearest             29.22  43.26   7.97  0.12  0.30  0.38  0.25  0.55         78.48  76.07  74.77  95.36
3 nearest, same state  28.95  39.22   7.59  0.12  0.31  0.44  0.28  0.55         79.96  74.58  76.99  95.36
state mean             31.03  41.09   6.97  0.12  0.26  0.41  0.34  0.53         77.74  75.70  78.66  93.88
India mean             41.91  69.47  10.59  0.27  0.00  0.00  0.00  0.00         71.80  35.62  61.78  91.84

For scale: the spread of the real district values (standard deviation) and how many districts sit in each class.

In [3]:
pd.DataFrame({'mean': soil[NUTRIENTS].mean(), 'sd': soil[NUTRIENTS].std(),
              **{c: [(cls(soil[n], n) == i).mean() * 100 for n in NUTRIENTS] for i, c in enumerate(['low/acidic %', 'medium/normal %', 'high/alkaline %'])}}).round(1)

,mean,sd,low/acidic %,medium/normal %,high/alkaline %
N,282.1,89.0,62.7,35.6,1.7
P,49.4,12.6,4.3,61.8,34.0
K,243.9,50.3,1.5,71.8,26.7
ph,7.2,0.4,8.2,91.8,0.0


**Are the filled districts as close to their neighbours as the test districts?** If the 84 districts the app fills
are farther from districts with data, the test above flatters the fill.

In [4]:
app = recommend._tables()[2]
filled = app[~app.district_code.isin(soil.district_code)]
flat, flon = np.radians(filled.lat.values)[:, None], np.radians(filled.lon.values)[:, None]
Df = 6371 * np.arccos(np.clip(np.sin(flat) * np.sin(lat) + np.cos(flat) * np.cos(lat) * np.cos(flon - lon), -1, 1))
dist = pd.DataFrame({
    'test districts (539)': pd.Series(np.sort(D, axis=1)[:, :3].mean(axis=1)).describe(),
    'filled districts (%d)' % len(filled): pd.Series(np.sort(Df, axis=1)[:, :3].mean(axis=1)).describe(),
}).loc[['count', '25%', '50%', '75%', 'max']].round(0)
print('mean distance to the 3 nearest districts with SHC data, km')
print(dist)
print('\nfilled districts by state:', filled.state_name.value_counts().head(8).to_dict())

mean distance to the 3 nearest districts with SHC data, km
       test districts (539)  filled districts (84)
count                 539.0                   84.0
25%                    45.0                   69.0
50%                    56.0                   87.0
75%                    71.0                  106.0
max                   712.0                  311.0

filled districts by state: {'Rajasthan': 28, 'Punjab': 19, 'Haryana': 18, 'Kerala': 12, 'Manipur': 4, 'Himachal Pradesh': 1, 'Nagaland': 1, 'Chandigarh': 1}


In [5]:
out = res.copy()
out.insert(0, 'date', '2026-11-04')
out.round(3).to_csv(RESULTS / '2026-11-04_soil_fill_check.csv', index=False)
print('saved')

saved


## Findings

Re-run Nov 4 with the card's own class limits (pH 6.5/8.2, P 28/56, K 140/280; see `data.SHC_LEVELS`). The first run
used pH 6.5/7.5, P 10/25, K 120/280; skill was the same, class match changed only because the limits changed.

1. **The 3-nearest fill is clearly better than one India-wide number** and about as good as anything simple:
   it removes 41% of the error for N, 25% for P, 32% for K and 56% for pH. Same-state neighbours or the state mean
   are level with it (state mean is a bit better for P); more neighbours (5, 10) don't help.
2. **Class match** (same card class as the real value): N 74%, P 76%, K 79%, pH 96%. But one number for all of India
   already matches 36%, 62%, 72% and 92%, because most districts sit in one class (N low, P medium, K medium, pH
   normal). The real gain is for N.
3. **The districts the app actually fills are farther from their neighbours** (median 87 km vs 56 km) and are whole
   states with little SHC data (Rajasthan, Punjab, Haryana, Kerala), so their error is likely larger than this test shows.
4. **pH barely varies between district averages** (sd 0.4): the card's "Normal" class spans 6.5–8.2 and holds 86% of
   India's samples, so class counts can't separate a pH 6.8 district from a pH 8.0 one.

Web check of filled districts against published studies: `research/notes/2026-11-04_soil_fill_check.md`.

## 2. Do the filled districts get the right crops?

The 84 filled districts have no SHC average, so none of them was in the crop model's training: a real out-of-sample
test. DES says what each grows. For each district: the app's top 3 (exactly what a farmer without a card sees, sown
Nov 4), compared with the district's main crop by area (among the model's 27 crops).

- **Fill:** soil from `recommend.area_soil` (3 nearest districts).
- **Fixed default soil:** one soil for everyone (N 250, P 40, K 210, pH 7.35), so only weather and location differ.
- **Popularity:** the 3 crops with the highest mean area share in the 539 training districts.

Metrics as in notebook 09: main crop first, main crop in the list, share of listed crops the district grows.

In [6]:
import datetime
import data

model_crops = recommend._model()['classes']
area = data.district_yield()
area = area[area.crop.isin(model_crops)]
grown = area.groupby('district_code').apply(lambda g: set(g.crop), include_groups=False)
main = area.loc[area.groupby('district_code').area_ha.idxmax()].set_index('district_code').crop
test = filled[filled.district_code.isin(main.index)].reset_index(drop=True)

train_area = area[area.district_code.isin(soil.district_code)]
shares = train_area.pivot_table(index='district_code', columns='crop', values='area_ha', fill_value=0)
popular = list(shares.div(shares.sum(axis=1), axis=0).mean().nlargest(3).index)
DEFAULT = (250, 40, 210, 7.35)
SOW = datetime.date(2026, 11, 4)

rows = []
for r in test.itertuples():
    fill, _ = recommend.area_soil(r.district_name, r.state_name)
    picks = {'fill': list(recommend.recommend(*fill.values(), r.district_name, r.state_name, sow=SOW)[1].crop),
             'fixed default soil': list(recommend.recommend(*DEFAULT, r.district_name, r.state_name, sow=SOW)[1].crop),
             'popularity': popular}
    for method, p in picks.items():
        rows.append({'state': r.state_name, 'district': r.district_name, 'method': method, 'picks': ', '.join(p),
                     'main': main[r.district_code], 'first': p[0] == main[r.district_code],
                     'in list': main[r.district_code] in p,
                     'grown': np.mean([c in grown[r.district_code] for c in p])})
picks = pd.DataFrame(rows)
print(len(test), 'filled districts with DES area; popularity =', popular)
picks.groupby('method')[['first', 'in list', 'grown']].mean().mul(100).round(1).rename(
    columns={'first': 'main crop 1st %', 'in list': 'main crop in list %', 'grown': 'picks grown %'})

84 filled districts with DES area; popularity = ['rice', 'wheat', 'maize']


,main crop 1st %,main crop in list %,picks grown %
method,,,
fill,48.8,84.5,99.6
fixed default soil,39.3,85.7,99.2
popularity,19.0,60.7,89.7


In [7]:
by_state = picks[picks.method == 'fill'].groupby('state').agg(
    districts=('district', 'size'), first=('first', 'mean'), in_list=('in list', 'mean'))
by_state[['first', 'in_list']] *= 100
print(by_state.round(0).sort_values('districts', ascending=False))
print()
print(picks[(picks.method == 'fill') & ~picks['in list']][['state', 'district', 'picks', 'main']].to_string(index=False))

                  districts  first  in_list
state                                      
Rajasthan                28   21.0     61.0
Punjab                   19   63.0    100.0
Haryana                  18   56.0     94.0
Kerala                   12   50.0     92.0
Manipur                   4  100.0    100.0
Chandigarh                1  100.0    100.0
Himachal Pradesh          1  100.0    100.0
Nagaland                  1  100.0    100.0

    state      district                    picks      main
Rajasthan         Ajmer    mustard, wheat, bajra     jowar
Rajasthan      Bhilwara  wheat, soybean, mustard     maize
Rajasthan       Bikaner    wheat, bajra, mustard mothbeans
Rajasthan         Churu    wheat, bajra, mustard  mungbean
Rajasthan    Ganganagar      wheat, rice, cotton   mustard
Rajasthan     Jaisalmer    mustard, wheat, bajra  chickpea
Rajasthan          Kota   wheat, blackgram, rice   soybean
Rajasthan        Nagaur    mustard, wheat, bajra  mungbean
Rajasthan          Pali    w

In [8]:
picks.to_csv(RESULTS / '2026-11-04_soil_fill_crop_picks.csv', index=False)
print('saved')

saved


## Findings, section 2

1. **In the 84 districts the app fills (none in training), the main crop is in the list 84.5% of the time and first
   48.8%**, against 60.7% and 19.0% for "rice, wheat, maize". Picks are almost always crops the district grows (99.6%).
2. **The filled soil helps the first pick, not the list:** with one fixed soil for everyone the list hit rate is the
   same (85.7%) but the main crop is first only 39.3%. Weather and location carry the list; soil sharpens the order.
3. **Punjab 100%, Haryana 94%, Kerala 92% in list; Rajasthan 61%.** Arid Rajasthan grows mostly moth bean, mung bean
   and jowar (DES), but no district with SHC data near it does, so the model offers wheat, bajra and mustard. This is a
   gap in training coverage, not in the fill.